In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('model/features_engineered.csv')
print(f"Загружено {len(df)} игр с {len(df.columns)} признаками\n")

Загружено 72629 игр с 96 признаками



In [3]:
# Определяем рекомендованные признаки на основе корреляционного анализа
recommended_features = [
    'elo_diff', 'home_elo_before', 'away_elo_before',
    
    'win_rate_diff_L20', 'win_rate_diff_L10', 'win_rate_diff_L5',
    'home_win_rate_L20', 'home_win_rate_L10', 'home_win_rate_L5',
    'away_win_rate_L20', 'away_win_rate_L10', 'away_win_rate_L5',
    
    'fg_pct_diff_L20', 'fg_pct_diff_L10', 'fg_pct_diff_L5',
    'fg3_pct_diff_L20', 'fg3_pct_diff_L10', 'fg3_pct_diff_L5',
    
    'h2h_home_win_rate', 'h2h_total_games',
    
    'season_progress',
]

# Проверяем какие признаки доступны
available_features = [f for f in recommended_features if f in df.columns]
missing_features = [f for f in recommended_features if f not in df.columns]

print(f"Доступно признаков: {len(available_features)}")
if missing_features:
    print(f"Отсутствуют признаки: {missing_features}")
print(f"\nИспользуемые признаки:\n{available_features}")

Доступно признаков: 21

Используемые признаки:
['elo_diff', 'home_elo_before', 'away_elo_before', 'win_rate_diff_L20', 'win_rate_diff_L10', 'win_rate_diff_L5', 'home_win_rate_L20', 'home_win_rate_L10', 'home_win_rate_L5', 'away_win_rate_L20', 'away_win_rate_L10', 'away_win_rate_L5', 'fg_pct_diff_L20', 'fg_pct_diff_L10', 'fg_pct_diff_L5', 'fg3_pct_diff_L20', 'fg3_pct_diff_L10', 'fg3_pct_diff_L5', 'h2h_home_win_rate', 'h2h_total_games', 'season_progress']


In [4]:
# Подготовка X и y
X = df[available_features].copy()
y = df['home_win'].copy()

# Удаляем строки с пропущенными значениями
mask = ~(X.isna().any(axis=1) | y.isna())
X = X[mask]
y = y[mask]

print(f"После удаления пропущенных значений: {len(X)} игр")
print(f"Распределение целевой переменной: {y.value_counts().to_dict()}")
print(f"Win rate домашних команд: {y.mean():.3f}\n")

После удаления пропущенных значений: 50843 игр
Распределение целевой переменной: {1: 29890, 0: 20953}
Win rate домашних команд: 0.588



In [5]:
# Разделяем данные хронологически (важно для временных рядов)
# 80% для обучения, 20% для тестирования
split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Обучающая выборка: {len(X_train)} игр")
print(f"Тестовая выборка: {len(X_test)} игр")
print(f"Train home win rate: {y_train.mean():.3f}")
print(f"Test home win rate: {y_test.mean():.3f}\n")

# Масштабируем признаки
print("Масштабирование признаков...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Масштабирование завершено\n")

Обучающая выборка: 40674 игр
Тестовая выборка: 10169 игр
Train home win rate: 0.598
Test home win rate: 0.547

Масштабирование признаков...
Масштабирование завершено



In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score
import numpy as np

# Задаем сетку гиперпараметров для поиска
param_dist = {
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': np.arange(2, 20),
    'min_samples_leaf': np.arange(1, 20),
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_features': ['auto', 'sqrt', 'log2', None]
}

dt = DecisionTreeClassifier(random_state=42)

rs = RandomizedSearchCV(
    dt, 
    param_distributions=param_dist,
    n_iter=100,
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rs.fit(X_train_scaled, y_train)
print("Лучшие параметры:", rs.best_params_)
print(f"Лучшая кросс-валидационная accuracy: {rs.best_score_:.4f}")

# Предсказания на тесте
y_pred = rs.predict(X_test_scaled)
print("\nОценка на тестовой выборке:")
print(f"F1 score: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC score: {roc_auc_score(y_test, y_pred):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Лучшие параметры: {'min_samples_split': 4, 'min_samples_leaf': 8, 'max_features': None, 'max_depth': 3, 'criterion': 'gini'}
Лучшая кросс-валидационная accuracy: 0.7746

Оценка на тестовой выборке:
F1 score: 0.6339
ROC AUC score: 0.6043
Accuracy: 0.6339


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Задаем сетку гиперпараметров для поиска
rf_param_dist = {
    'n_estimators': np.arange(50, 301, 50),
    'max_depth': [None] + list(np.arange(2, 21, 2)),
    'min_samples_split': np.arange(2, 11, 2),
    'min_samples_leaf': np.arange(1, 11, 2),
    'criterion': ['gini', 'entropy', 'log_loss'],
}

rf = RandomForestClassifier(random_state=42)

rf_rs = RandomizedSearchCV(
    rf,
    param_distributions=rf_param_dist,
    n_iter=1,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

rf_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (Random Forest):", rf_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {rf_rs.best_score_:.4f}")

# Предсказания на тестовой выборке
rf_y_pred = rf_rs.predict(X_test_scaled)
rf_y_proba = rf_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(rf_rs, "predict_proba") else None

In [39]:
print("\nОценка на тестовой выборке (Random Forest):")
print(f"F1 score: {f1_score(y_test, rf_y_pred):.4f}")
if rf_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, rf_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, rf_y_pred):.4f}")


Оценка на тестовой выборке (Random Forest):
F1 score: 0.7225
ROC AUC score: 0.7140
Accuracy: 0.6618


Градиентный бустинг не получится обучить за один раз -- слишком долго. Сначала подберем для двух гиперпараметров, потом для остальных.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Задаем сетку гиперпараметров для поиска
gb_param_dist = {
    'n_estimators': np.arange(50, 301, 50),
    'max_depth': np.arange(2, 11, 2),
    # 'learning_rate': np.linspace(0.01, 0.3, 5),
    # 'subsample': np.linspace(0.6, 1.0, 3),
    # 'min_samples_split': np.arange(2, 11, 2),
    # 'min_samples_leaf': np.arange(1, 11, 2),
}

gb = GradientBoostingClassifier(random_state=42, verbose=3)

gb_rs = RandomizedSearchCV(
    gb,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    verbose=3,  # Вывод процесса обучения
    random_state=42,
    n_jobs=-1
)

gb_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (Gradient Boosting):", gb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {gb_rs.best_score_:.4f}")

# Предсказания на тестовой выборке
gb_y_pred = gb_rs.predict(X_test_scaled)
gb_y_proba = gb_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(gb_rs, "predict_proba") else None

In [38]:
print("\nОценка на тестовой выборке (Gradient Boosting):")
print(f"F1 score: {f1_score(y_test, gb_y_pred):.4f}")
if gb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, gb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, gb_y_pred):.4f}")


Оценка на тестовой выборке (Gradient Boosting):
F1 score: 0.7269
ROC AUC score: 0.7001
Accuracy: 0.5892


In [19]:
gb_rs.best_params_

{'n_estimators': 50, 'max_depth': 2}

In [ ]:
gb = GradientBoostingClassifier(random_state=42, verbose=3, n_estimators=50, max_depth=2)

gb_param_dist = {
    'learning_rate': np.linspace(0.01, 0.3, 5),
    'min_samples_split': np.arange(2, 11, 2),
    'min_samples_leaf': np.arange(1, 11, 2),
}

gb_rs = RandomizedSearchCV(
    gb,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-3
)

gb_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (Gradient Boosting):", gb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {gb_rs.best_score_:.4f}")

# Предсказания на тестовой выборке
gb_y_pred = gb_rs.predict(X_test_scaled)
gb_y_proba = gb_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(gb_rs, "predict_proba") else None

In [37]:
print("\nОценка на тестовой выборке (Gradient Boosting):")
print(f"F1 score: {f1_score(y_test, gb_y_pred):.4f}")
if gb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, gb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, gb_y_pred):.4f}")


Оценка на тестовой выборке (Gradient Boosting):
F1 score: 0.7269
ROC AUC score: 0.7001
Accuracy: 0.5892


In [ ]:
!pip3 install catboost

In [ ]:
from catboost import CatBoostClassifier

catboost = CatBoostClassifier(
    random_seed=42,
    verbose=2,
    thread_count=-1
)

# Гиперпараметры для RandomizedSearchCV
catboost_param_dist = {
    'iterations': [100, 200, 300, 400],
    'depth': [3, 4, 5, 6],
    'learning_rate': np.linspace(0.01, 0.3, 5),
    'l2_leaf_reg': [1, 3, 5, 7, 9, 11],
}

catboost_rs = RandomizedSearchCV(
    catboost,
    param_distributions=catboost_param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

catboost_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (CatBoost):", catboost_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {catboost_rs.best_score_:.4f}")

cb_y_pred = catboost_rs.predict(X_test_scaled)
cb_y_proba = catboost_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(catboost_rs, "predict_proba") else None


In [34]:
print("\nОценка на тестовой выборке (CatBoost):")
print(f"F1 score: {f1_score(y_test, cb_y_pred):.4f}")
if cb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, cb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, cb_y_pred):.4f}")


Оценка на тестовой выборке (CatBoost):
F1 score: 0.7248
ROC AUC score: 0.7172
Accuracy: 0.6620


In [ ]:
!pip3 install lightgbm

In [ ]:
!brew install libomp

In [ ]:
from lightgbm import LGBMClassifier

lgb_param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [2, 4, 6, 8, -1],
    'learning_rate': np.linspace(0.01, 0.3, 5),
    'num_leaves': [15, 31, 63, 127],
    'min_child_samples': [5, 10, 20]
}

lgb = LGBMClassifier(random_state=42, n_jobs=-1)

lgb_rs = RandomizedSearchCV(
    lgb,
    param_distributions=lgb_param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

lgb_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (LightGBM):", lgb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {lgb_rs.best_score_:.4f}")

lgb_y_pred = lgb_rs.predict(X_test_scaled)
lgb_y_proba = lgb_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(lgb_rs, "predict_proba") else None

In [35]:
print("\nОценка на тестовой выборке (LightGBM):")
print(f"F1 score: {f1_score(y_test, lgb_y_pred):.4f}")
if lgb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, lgb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, lgb_y_pred):.4f}")


Оценка на тестовой выборке (LightGBM):
F1 score: 0.7288
ROC AUC score: 0.6914
Accuracy: 0.6177


In [32]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV

# Параметры для RandomizedSearchCV
knn_param_dist = {
    'n_neighbors': list(range(1, 31)),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1 — манхэттен, 2 — евклид
}

knn = KNeighborsClassifier(n_jobs=-1)

knn_rs = RandomizedSearchCV(
    knn,
    param_distributions=knn_param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

knn_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (KNN):", knn_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {knn_rs.best_score_:.4f}")

knn_y_pred = knn_rs.predict(X_test_scaled)
knn_y_proba = knn_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(knn_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (KNN):")
print(f"F1 score: {f1_score(y_test, knn_y_pred):.4f}")
if knn_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, knn_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, knn_y_pred):.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Лучшие параметры (KNN): {'weights': 'distance', 'p': 2, 'n_neighbors': 27}
Лучшая кросс-валидационная f1: 0.7553

Оценка на тестовой выборке (KNN):
F1 score: 0.7090
ROC AUC score: 0.6953
Accuracy: 0.6512


In [ ]:
from xgboost import XGBClassifier

# Параметры для RandomizedSearchCV для XGBoost
xgb_param_dist = {
    'n_estimators': [30, 50, 75, 100, 150],
    'max_depth': [2, 3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)

xgb_rs = RandomizedSearchCV(
    xgb,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

xgb_rs.fit(X_train, y_train)
print("Лучшие параметры (XGBoost):", xgb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {xgb_rs.best_score_:.4f}")

xgb_y_pred = xgb_rs.predict(X_test)
xgb_y_proba = xgb_rs.predict_proba(X_test)[:, 1] if hasattr(xgb_rs, "predict_proba") else None

In [36]:
print("\nОценка на тестовой выборке (XGBoost):")
print(f"F1 score: {f1_score(y_test, xgb_y_pred):.4f}")
if xgb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, xgb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, xgb_y_pred):.4f}")


Оценка на тестовой выборке (XGBoost):
F1 score: 0.7325
ROC AUC score: 0.7096
Accuracy: 0.6483


Вывод: по итогам обучения и поиска различных гиперпараметров с помощью RandomizedSearchCV получили лучший результат по f1 score при XGBoost: 0.7325, accuracy при этом была лучшей при Catboost: 0.662.